# Vector stores and semantic search



In [1]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Part I: Basic vector store implementation

In [8]:
class Document:
  def __init__(self, text: str, metadata: dict[str, str]):
    self.text = text
    self.metadata = metadata


class SearchResult:
  def __init__(self, score: float, document: Document):
    self.score = score
    self.document = document


class VectorStore:
  def __init__(self, embedding_model: SentenceTransformer):
    self.embedding_model = embedding_model
    self.documents = []
    self.embeddings = []

  def add_documents(self, documents: list[Document]):
    self.documents.extend(documents)
    texts = [doc.text for doc in documents]
    new_embeddings = self.embedding_model.encode(texts)
    self.embeddings.extend(new_embeddings)


  def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
    query_embedding = self.embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    embeddings_matrix = np.array(self.embeddings)

    # Cos calculo
    similarities = cosine_similarity(
        query_embedding,
        embeddings_matrix
    )[0]

    # Ordenados indice y resultados
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for idx in top_indices:
      results.append(
          SearchResult(
              score=float(similarities[idx]),
              document=self.documents[idx]
          )
      )

    return results

In [9]:
df = pd.read_csv("animal-fun-facts-dataset.csv")
print(df)

           animal_name                                             source  \
0             aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
1             aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
2             aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
3             aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
4             aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
...                ...                                                ...   
7729  yellow rat snake  https://seaworld.org/animals/facts/reptiles/ye...   
7730  yellow rat snake  https://seaworld.org/animals/facts/reptiles/ye...   
7731  yellow rat snake  https://seaworld.org/animals/facts/reptiles/ye...   
7732  yellow rat snake  https://seaworld.org/animals/facts/reptiles/ye...   
7733  yellow rat snake  https://seaworld.org/animals/facts/reptiles/ye...   

                                                   text media_link  \
0    

In [10]:
# Como una lista de instancias
documents = []

for _, row in df.iterrows():

    # De la clase Document proporcionada
    doc = Document(
        # Columna text es texto del documento
        text=str(row["text"]),

        # animal_name, source, media_link y wikipedia_link son metadatos
        metadata = {
        "animal_name": str(row["animal_name"]),
        "source": str(row["source"]),
        "media_link": str(row["media_link"]),
        "wikipedia_link": str(row["wikipedia_link"])
        }
    )

    documents.append(doc)

print(f"Total documentos: {len(documents)}")

Total documentos: 7734


In [11]:
# Crear una instancia de VectorStore y agregar los documentos
model = SentenceTransformer('all-MiniLM-L6-v2')
vs_animals = VectorStore(model)
vs_animals.add_documents(documents)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
# Realiza al menos 5 consultas de ejemplo y muestra los resultados obtenidos
queries = [
    "Animals that can fly",
    "Animals that live in water",
    "Mammals animals",
    "Animals that are carnivores",
    "Animals that use camouflage"
]

for q in queries:
  print("----------------------------------------------------------------------------")
  print(f"QUERY: {q}")
  results = vs_animals.search(q, top_k=3)

  # Incluye el score, texto y metadatos
  for r in results:
    print("Score:", r.score)
    print("Text:", r.document.text)
    print("Metadata:", r.document.metadata)
    print()

----------------------------------------------------------------------------
QUERY: Animals that can fly
Score: 0.7076399326324463
Text: They don’t fly, they glide.
The only mammal which can independently fly is the bat. Instead, colugas glide which works in the same way as a wingsuit.
Metadata: {'animal_name': 'colugo (flying lemur)', 'source': 'https://factanimal.com/colugo/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Colugo'}

Score: 0.6900621056556702
Text: Not all birds are able to fly!
Metadata: {'animal_name': 'bird', 'source': 'https://a-z-animals.com/animals/bird/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Bird'}

Score: 0.6480565667152405
Text: They rarely fly..
They move around on foot most of the time, only taking to the air to reach their nests or for courtship displays.
Metadata: {'animal_name': 'secretary bird', 'source': 'https://factanimal.com/secretarybird/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Secretarybird'}

--------------------------------------

## Part II: Filtering by metadata

In [ ]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass